In [2]:
import os

# mount drive if colab
from google.colab import drive
drive.mount('/content/gdrive')
os.chdir("/content/gdrive/MyDrive/Soil_Moisture/Local_Training")

Mounted at /content/gdrive


In [ ]:
import os
import numpy as np
import tensorflow as tf


# OUTPUT_NAME = "Fine_Tuning_Osiris" 
# ROOT_DIR = "/content/gdrive/MyDrive/Soil_Moisture/dataset_training" # Si on Colab
# OSIRIS_DIR = os.path.join(ROOT_DIR, "Osiris_unified") # Si on Colab

# drive_dir = os.path.join("/content/gdrive/MyDrive/Soil_Moisture/outputs", OUTPUT_NAME) # Si on Colab
# RESULTS_CSV_PATH = os.path.join(drive_dir, "results.csv") # Si on Colab

# # Surcharge des chemins lus par config.py (via variables d'environnement).
# # À définir AVANT l'import de Fcn_Training (cellule suivante).
# os.environ["ROOT_DIR"] = ROOT_DIR
# os.environ["OSIRIS_DIR_UNIFIED"] = OSIRIS_DIR
# os.environ["OUTPUT_NAME"] = OUTPUT_NAME
# os.environ["DRIVE_DIR"] = drive_dir


FOLDER_NAME = "Fine_Tuning" 

ROOT_DIR = "/home/theo/Dataset" # Si on local machine

OSIRIS_DIR = os.path.join(ROOT_DIR, "Osiris_dataset")

drive_dir = os.path.join("/home/theo/Documents", FOLDER_NAME, "outputs") # Si on local machine

RESULTS_CSV_PATH = os.path.join(drive_dir, "results.csv") # Si on local machine

# ============================================================
# Configuration
# ============================================================

FOLDER_ISMN = "station_depth_csv"
MASTER_CSV_PATH = os.path.join(ROOT_DIR, FOLDER_ISMN, "Soil_Properties_Master.csv")

base_path = os.path.join(ROOT_DIR, FOLDER_ISMN, "depth")

FULL_DENSE = [  "ET0", "IRRAD", "TMIN", "TMAX", "VAP", "WIND", "RAIN",
              "VPD", "T_RANGE",
              "RAIN_CUM_3D", "RAIN_CUM_7D", "RAIN_CUM_14D",
              "doy_sin", "doy_cos"]

FULL_SOIL  = [ "clay", "silt", "bulk", "sand", "dem", "ksat_m_1km", "dem_slope", "dem_aspect", "dem_twi"]

FULL_SPARSE = ["S2_B2", "S2_B3", "S2_B4", "S2_B5", "S2_B6", "S2_B7",
               "S2_B8", "S2_B8A", "S2_B11", "S2_B12",
               "S2_NDVI", "S2_NDWI", "S2_SAVI", "S2_MNDWI", "S2_NBR",
               "S1_VV", "S1_VH", "S1_angle",
               "S1_VV_over_VH", "S1_VH_over_VV",
               "HLS_B2", "HLS_B3", "HLS_B4", "HLS_B5", 
                     "HLS_B6", "HLS_B7", "HLS_B9", "HLS_B10", "HLS_B11", "HLS_NDVI"]

FULL_SPARSE_S1 = ["S1_VV", "S1_VH", "S1_angle",
                   "S1_VV_over_VH", "S1_VH_over_VV"]

FULL_SPARSE_S2 = ["S2_B2", "S2_B3", "S2_B4", "S2_B5", "S2_B6", "S2_B7",
               "S2_B8", "S2_B8A", "S2_B11", "S2_B12",
               "S2_NDVI", "S2_NDWI", "S2_SAVI", "S2_MNDWI", "S2_NBR"]

FULL_SPARSE_HLS30 = ["HLS_B2", "HLS_B3", "HLS_B4", "HLS_B5", 
                     "HLS_B6", "HLS_B7", "HLS_B9", "HLS_B10", "HLS_B11", "HLS_NDVI"]

# Training settings
HORIZONS = [7]              # predict * days ahead 
LOOKBACK = [7]                  # use past * days to predict next day
DEPTHS = [ 0.1 , 0.2 , 0.3 ]#, 0.4, 0.5] # 0.5]
        #    0.3, 0.4, 0.5]  # we will loop over these depths and train one model per depth
ALL_NETWORKS = [
    "all"]

    # ["COSMOS-UK", "GROW", "PTSMN", "TAHMO", "TERENO"]
SEPARATE_NETWORKS = [

     ["COSMOS-UK"],
             ["DWD"],
             ["FR_Aqui", "GROW"],
             ["PTSMN"], 
            ["SMOSMANIA"], 
            ["SOILSCAPE"],
            ["TAHMO","TERENO"],
        ["TERENO"], 
            ["TWENTE"], 
            ["XMS-CAT"]
        ]

NETWORK1 = [
            ["TAHMO","TERENO","FR_Aqui"],
            ["TAHMO","TERENO","GROW"],
            ["TAHMO","TERENO","GROW","FR_Aqui"]
        ]


NB_WINDOWS = [100000]    
# Models: xgboost and lightgbm are fast, non-DL
MODELS = ["lstm"]
# , "lightgbm"]

def _without(lst, *items):
    return [x for x in lst if x not in items]

FEATURE_CONFIGS = [


    # # ── 4.2.1 Comparaison des modèles ──
    # {"name": "Models", "dense": FULL_DENSE, "soil": FULL_SOIL, "sparse": FULL_SPARSE,
    #  "lookbacks": [7], "horizons": [7], "depths": DEPTHS, "nb_windows": [50000],
    #  "models": ["xgboost", "lightgbm", "lstm", "gru", "tcn", "transformer"],
    #  "networks": ALL_NETWORKS},

#     # ── 4.2.2 Effet du lookback ──
#     {"name": "Lookback", "dense": FULL_DENSE, "soil": FULL_SOIL, "sparse": FULL_SPARSE_S1,
#      "lookbacks": [1, 2, 4, 7, 14, 21, 28],
#      "horizons": [7], "depths": DEPTHS, "nb_windows": [50000],
#      "models": ["xgboost", "lstm"],
#      "networks": ALL_NETWORKS},

#     # ── 4.2.3 Effet de l'horizon ──
#     {"name": "Horizon", "dense": FULL_DENSE, "soil": FULL_SOIL, "sparse": FULL_SPARSE_S1,
#      "lookbacks": [7],
#      "horizons": [1, 3, 7, 14, 21],
#      "depths": DEPTHS, "nb_windows": [50000],
#      "models": ["xgboost", "lstm"],
#      "networks": ALL_NETWORKS},

#     # ── 4.2.4 Effet du nombre de fenêtres ──
#     {"name": "Nb_Windows", "dense": FULL_DENSE, "soil": FULL_SOIL, "sparse": FULL_SPARSE_S1,
#      "lookbacks": [7], "horizons": [7], "depths": DEPTHS,
#      "nb_windows": [1000, 5000, 10000, 20000, 50000, 100000],
#      "models": ["xgboost", "lstm"],
#      "networks": ALL_NETWORKS},

    {"name": "Fine_Tuning", "dense": FULL_DENSE, "soil": FULL_SOIL, "sparse": FULL_SPARSE,
     "lookbacks": [7], "horizons": [7], "depths": DEPTHS, "nb_windows": [50000],
        "models": ["lstm"],
        "networks": ALL_NETWORKS},

#     # ── 4.2.6 Importance de la proximité géographique (par réseau) ──
#     {"name": "Networks", "dense": FULL_DENSE, "soil": FULL_SOIL, "sparse": FULL_SPARSE_S1,
#      "lookbacks": [7], "horizons": [7], "depths": DEPTHS, "nb_windows": [50000],
#      "models": ["xgboost"],
#      "networks": SEPARATE_NETWORKS},
]

SAVE_PLOTS = True
SAVE_NETWORKS_DIR = True
SAVE_MODELS_DIR = True
SAVE_RESULTS_CSV = True



TARGET_COL = "soil_moisture"
DATE_COL = "date_time"  # L'index temporel est sauvegardé sous date_time par process_timeseries
EPOCHS = 150
BATCH_SIZE = 32
SEED = 8

if SAVE_MODELS_DIR:
    os.makedirs(drive_dir, exist_ok=True)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# MONTHS=[4,5,6,7,8,9]
MONTHS = None

In [ ]:
from Fcn_Training import (full_training, 
                        get_osiris_data,
                        osiris_train,
                        osiris_fine_tuning,
                        full_eval_osiris
                        )

In [5]:
# # 1. Charger les données de test Grandvillers
# test_list_local = Get_Grandvillers_data(os.path.join(Grandvillers_path, "Grandvillers_satellites"))
# # 2. Préparer directement la version de test ERA5 pour chaque fichier
# test_list_era5 = []
# print("--- PRÉPARATION DU JEU DE TEST ERA5 ---")
# for i, df in enumerate(test_list_local):
#     df_substituted = prepare_era5_dataset(df, display_name=f"Fichier {i}")
#     test_list_era5.append(df_substituted)

In [6]:
all_dfs = get_osiris_data(OSIRIS_DIR)

In [7]:
# for feat_cfg in FEATURE_CONFIGS:
#     feature_cols = feat_cfg["dense"] + feat_cfg["soil"] + feat_cfg["sparse"]
#     osiris_train(feat_cfg, drive_dir, all_dfs)

In [ ]:
### Training
for feat_cfg in FEATURE_CONFIGS:
    # Réassigner les globales
    feature_cols = feat_cfg["dense"] + feat_cfg["soil"] + feat_cfg["sparse"]
    print(f"\n========== FEATURE SET: {feat_cfg['name']} ==========")
    print(f"  Features: {feature_cols}")

    ############### Training ISMN ###############
    full_training(feat_cfg, base_path, drive_dir, MONTHS)

    # ########### Evaluation on Osiris ###############
    full_eval_osiris(feat_cfg, drive_dir, all_dfs)

    # ############ Fine-Tuning Osiris ###############
    osiris_fine_tuning(feat_cfg, drive_dir, all_dfs)


========== FEATURE SET: Fine_Tuning ==========
  Features: ['ET0', 'IRRAD', 'TMIN', 'TMAX', 'VAP', 'WIND', 'RAIN', 'VPD', 'T_RANGE', 'RAIN_CUM_3D', 'RAIN_CUM_7D', 'RAIN_CUM_14D', 'doy_sin', 'doy_cos', 'clay', 'silt', 'bulk', 'sand', 'dem', 'ksat_m_1km', 'dem_slope', 'dem_aspect', 'dem_twi', 'S2_B2', 'S2_B3', 'S2_B4', 'S2_B5', 'S2_B6', 'S2_B7', 'S2_B8', 'S2_B8A', 'S2_B11', 'S2_B12', 'S2_NDVI', 'S2_NDWI', 'S2_SAVI', 'S2_MNDWI', 'S2_NBR', 'S1_VV', 'S1_VH', 'S1_angle', 'S1_VV_over_VH', 'S1_VH_over_VV', 'HLS_B2', 'HLS_B3', 'HLS_B4', 'HLS_B5', 'HLS_B6', 'HLS_B7', 'HLS_B9', 'HLS_B10', 'HLS_B11', 'HLS_NDVI']

====================== NEW DEPTH: 0.1 ======================
Stations: train=42, val=9, test=10
Fichiers chargés: train=115, val=19, test=19
Spatial split: train=115 files, val=19 files, test=19 files
--- Lookback: 7 ---
--- Horizon: 7 ---

=== Training with NB=50000 windows ===

      > Modèle: lstm
Train windows: 40000, Val windows: 10000, (lookback=7)
Epoch 1/150
1250/1250 ━━━━━━━━━━━